# SQLite With Python

## Quick notes
### What this covers
- Using Python's built-in `sqlite3` module as a SQL playground: parameters, cursors, row factories, type adapters.
- Teaching SQLite new tricks from Python: custom scalar, aggregate and window functions, collations.
- Watching and guarding SQL: trace callback, authorizer, progress handler.
- Transactions from Python (`with con:`, `isolation_level`, `autocommit`), errors, threads, blobs, pandas.
- This notebook is the SQL side. For the general "database access in Python" story, see `01-GeneralPurposeLanguage/Python/11-Applied-Python/05-SQLite-With-Python` (not repeated here).
### Key points
- `con.execute(sql, params)` runs one statement. Always pass values as parameters (`?` or `:name`), never paste them into the SQL string.
- A cursor is an iterator: `for row in cur` streams rows. `fetchone`, `fetchmany(n)`, `fetchall` pull them in chunks.
- `row_factory = sqlite3.Row` gives name access (`row["name"]`). A function of your own gives dicts or anything else.
- `create_function` lets SQL call Python. Mark it `deterministic=True` when the same input always gives the same output.
- `with con:` commits on success and rolls back on an exception. It does NOT close the connection.
### Gotchas (the part worth revising twice)
- Parameters can only stand for values, not for table or column names. Names need a whitelist, not string pasting.
- `with con:` does not close the connection. Use `contextlib.closing(con)` or call `con.close()`.
- `executemany` wants a sequence of tuples. A single tuple of values is a different (wrong) call.
- A Python exception inside a user-defined function becomes a generic `OperationalError` in SQL.
- Default type handling is thin: dates come back as text unless you register converters and pass `detect_types`.
- A connection belongs to the thread that made it unless you pass `check_same_thread=False` (and then you must serialise access yourself).
- `executescript` first COMMITs any open transaction and runs in autocommit style. Do not use it mid-transaction.
- Some features (JSON, FTS5, math functions) depend on how SQLite was compiled. Guard demos with a try/except if code must run everywhere.
### When it matters
- Prototyping and testing SQL ideas quickly, with real Python data around it.
- Small tools, scripts, ETL steps, test fixtures, local app storage.
- Anywhere a query needs logic SQL itself lacks (a regex match, a custom score).
### One-line recap example
- `con.execute("SELECT name FROM people WHERE age > ?", (30,)).fetchall()` means "ask the database safely, get a list of tuples back".

## 01. Setup and a tiny dataset
- The first cell of this notebook already made `con`, an in-memory database, and a `%%sql` magic.
- `%%sql` cells use that same `con`, so Python cells and SQL cells see the same tables.
- Here we make a small `people` table with SQL and then query it from Python.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE people (
    id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT, age INTEGER, joined TEXT
);
INSERT INTO people VALUES
    (1,'Asha','Pune',34,'2023-01-15'),
    (2,'Bilal','Delhi',28,'2023-03-02'),
    (3,'Chen','Pune',41,'2022-11-20'),
    (4,'Dara',NULL,25,'2024-02-10'),
    (5,'Elif','Delhi',37,'2024-05-01');
SELECT * FROM people ORDER BY id;

id | name  | city  | age | joined    
---+-------+-------+-----+-----------
1  | Asha  | Pune  | 34  | 2023-01-15
2  | Bilal | Delhi | 28  | 2023-03-02
3  | Chen  | Pune  | 41  | 2022-11-20
4  | Dara  | NULL  | 25  | 2024-02-10
5  | Elif  | Delhi | 37  | 2024-05-01
(5 rows)



In [3]:
cur = con.execute("SELECT id, name, city FROM people ORDER BY id")
print(type(cur).__name__)
print(cur.description[0][0], [d[0] for d in cur.description])
print(cur.fetchall())

Cursor
id ['id', 'name', 'city']
[(1, 'Asha', 'Pune'), (2, 'Bilal', 'Delhi'), (3, 'Chen', 'Pune'), (4, 'Dara', None), (5, 'Elif', 'Delhi')]


## 02. Parameters: qmark and named
- `?` placeholders take a tuple (or list), in order.
- `:name` placeholders take a dict, by key. Easier to read when there are many values, and one name can be used twice.
- The driver sends the value separately from the SQL text, so quotes in the value cannot break the query.

In [4]:
print(con.execute("SELECT name FROM people WHERE city = ? AND age > ? ORDER BY name", ("Pune", 30)).fetchall())
print(con.execute("SELECT name FROM people WHERE age BETWEEN :lo AND :hi ORDER BY name", {"lo": 28, "hi": 37}).fetchall())
print(con.execute("SELECT :x + :x", {"x": 21}).fetchone())

[('Asha',), ('Chen',)]
[('Asha',), ('Bilal',), ('Elif',)]
(42,)


## 03. SQL injection: bad vs good
- Bad: build the SQL by gluing text together. Input like `x' OR '1'='1` changes the meaning of the query.
- Good: pass the value as a parameter. The same input is just a strange name that matches nothing.
- Parameters cannot replace identifiers (table or column names). For those, check against a fixed whitelist.

In [5]:
evil = "x' OR '1'='1"
bad_sql = f"SELECT name FROM people WHERE name = '{evil}' ORDER BY name"
print(bad_sql)
print("bad :", con.execute(bad_sql).fetchall())          # leaks every row
print("good:", con.execute("SELECT name FROM people WHERE name = ? ORDER BY name", (evil,)).fetchall())

ALLOWED_SORT = {"name", "age", "joined"}
def sorted_people(col):
    if col not in ALLOWED_SORT:
        raise ValueError(f"cannot sort by {col!r}")
    return [r[0] for r in con.execute(f"SELECT name FROM people ORDER BY {col}, id")]
print(sorted_people("age"))
try:
    sorted_people("age; DROP TABLE people")
except ValueError as e:
    print("blocked:", e)

SELECT name FROM people WHERE name = 'x' OR '1'='1' ORDER BY name
bad : [('Asha',), ('Bilal',), ('Chen',), ('Dara',), ('Elif',)]
good: []
['Dara', 'Bilal', 'Asha', 'Elif', 'Chen']
blocked: cannot sort by 'age; DROP TABLE people'


## 04. executemany and executescript
- `executemany(sql, rows)` runs one statement once per row. Fast and tidy for bulk inserts.
- `executescript(text)` runs many statements in one string. It takes no parameters.
- Use `executemany` for data, `executescript` for schema setup.

In [6]:
con.executescript("""
CREATE TABLE orders (id INTEGER PRIMARY KEY, person_id INTEGER, item TEXT, qty INTEGER, price REAL);
CREATE INDEX idx_orders_person ON orders(person_id);
""")
rows = [(1, "pen", 10, 1.5), (1, "book", 2, 12.0), (2, "pen", 5, 1.5),
        (3, "lamp", 1, 30.0), (3, "pen", 20, 1.4), (5, "book", 1, 12.5)]
cur = con.executemany("INSERT INTO orders (person_id, item, qty, price) VALUES (?, ?, ?, ?)", rows)
print("inserted:", cur.rowcount)
print(con.execute("SELECT COUNT(*), SUM(qty) FROM orders").fetchone())

inserted: 6
(6, 39)


## 05. Fetching: fetchone, fetchmany, fetchall, and cursors as iterators
- `fetchone()` returns one row, or `None` when the rows run out.
- `fetchmany(n)` returns up to `n` rows. Good for processing big results in batches.
- A cursor is also an iterator, so `for row in cur` streams rows without building a big list.
- Rows can be read only once. A cursor that is used up is empty, so re-run the query to read again.

In [7]:
cur = con.execute("SELECT id, name FROM people ORDER BY id")
print(cur.fetchone())
print(cur.fetchmany(2))
print([r for r in cur])
print(cur.fetchone())          # exhausted
cur.execute("SELECT name FROM people ORDER BY id")
print([name for (name,) in cur])

(1, 'Asha')
[(2, 'Bilal'), (3, 'Chen')]
[(4, 'Dara'), (5, 'Elif')]
None
['Asha', 'Bilal', 'Chen', 'Dara', 'Elif']


## 06. row_factory: sqlite3.Row and dicts
- Default rows are plain tuples. Reading by position is fragile when the column list changes.
- `sqlite3.Row` supports `row["name"]`, `row.keys()` and still works like a tuple.
- Your own factory can return dicts, named tuples, or dataclasses.
- `row_factory` is set per connection or per cursor.

In [8]:
cur = con.cursor()
cur.row_factory = sqlite3.Row
r = cur.execute("SELECT id, name, age FROM people WHERE id = 1").fetchone()
print(r["name"], r[2], r.keys(), tuple(r))

def dict_factory(cursor, row):
    return {d[0]: v for d, v in zip(cursor.description, row)}

cur2 = con.cursor()
cur2.row_factory = dict_factory
print(cur2.execute("SELECT id, name FROM people ORDER BY id LIMIT 2").fetchall())

Asha 34 ['id', 'name', 'age'] (1, 'Asha', 34)
[{'id': 1, 'name': 'Asha'}, {'id': 2, 'name': 'Bilal'}]


## 07. Adapters, converters and detect_types
- SQLite only stores NULL, INTEGER, REAL, TEXT, BLOB. Other Python types need translating.
- An adapter turns a Python object into a storable value. A converter turns stored bytes back into a Python object.
- Converters run when the declared column type matches (`detect_types=PARSE_DECLTYPES`) or the column alias matches (`PARSE_COLNAMES`).
- Python 3.12 deprecates the built-in date/datetime adapters, so register your own (shown below).

In [9]:
import datetime, json
sqlite3.register_adapter(datetime.date, lambda d: d.isoformat())
sqlite3.register_converter("DATE", lambda b: datetime.date.fromisoformat(b.decode()))
sqlite3.register_adapter(dict, json.dumps)
sqlite3.register_converter("JSONDOC", lambda b: json.loads(b.decode()))

m = sqlite3.connect(":memory:", detect_types=sqlite3.PARSE_DECLTYPES)
m.execute("CREATE TABLE ev (d DATE, meta JSONDOC)")
m.execute("INSERT INTO ev VALUES (?, ?)", (datetime.date(2024, 3, 9), {"tags": ["a", "b"], "n": 2}))
d, meta = m.execute("SELECT d, meta FROM ev").fetchone()
print(type(d).__name__, d, "|", type(meta).__name__, meta["tags"])
print(m.execute("SELECT typeof(d), d FROM ev").fetchone())    # stored as plain TEXT
m.close()

date 2024-03-09 | dict ['a', 'b']
('text', datetime.date(2024, 3, 9))


## 08. create_function: SQL calls Python (scalar)
- `con.create_function(name, n_args, func)` makes a Python function callable from SQL.
- `deterministic=True` promises the same input gives the same output. SQLite can then use it in indexes and cache it.
- Return types are limited to None, int, float, str, bytes. Return `None` for NULL.
- NULL arrives in Python as `None`, so handle it.

In [10]:
import re
def initials(name):
    if name is None:
        return None
    return "".join(w[0].upper() for w in name.split())

def regexp(pattern, value):                       # powers the REGEXP operator
    return value is not None and re.search(pattern, value) is not None

con.create_function("initials", 1, initials, deterministic=True)
con.create_function("regexp", 2, regexp, deterministic=True)
print(con.execute("SELECT name, initials(name || ' Khan') FROM people ORDER BY id LIMIT 2").fetchall())
print(con.execute("SELECT name FROM people WHERE name REGEXP '^[A-C]' ORDER BY name").fetchall())
print(con.execute("SELECT initials(NULL)").fetchone())

[('Asha', 'AK'), ('Bilal', 'BK')]
[('Asha',), ('Bilal',), ('Chen',)]
(None,)


## 09. Errors inside user functions
- If your Python function raises, SQL sees `sqlite3.OperationalError: user-defined function raised exception`. The original message is lost unless you log it.
- Turn on `sqlite3.enable_callback_tracebacks(True)` to print the real traceback to stderr while debugging.
- Safer pattern: catch inside the function and return `None`.

In [11]:
def risky(x):
    return 10 // x

def safe(x):
    try:
        return 10 // x
    except ZeroDivisionError:
        return None

con.create_function("risky", 1, risky)
con.create_function("safe", 1, safe)
try:
    con.execute("SELECT risky(0)").fetchone()
except sqlite3.OperationalError as e:
    print(type(e).__name__, "-", e)
print(con.execute("SELECT safe(0), safe(5)").fetchone())

OperationalError - user-defined function raised exception
(None, 2)


## 10. create_aggregate and create_window_function
- An aggregate is a class with `step(value)` and `finalize()`. SQL then uses it with `GROUP BY` like `SUM`.
- A window function class adds `value()` (current result) and `inverse(value)` (drop a row leaving the frame). That lets it work with `OVER (... ROWS BETWEEN ...)`.
- Use these for things SQLite has no built-in for, like a median or a string product.
- For window frames themselves, see `03-Window-Functions/01-Window-Functions`.

In [12]:
class Median:
    def __init__(self):
        self.vals = []
    def step(self, v):
        if v is not None:
            self.vals.append(v)
    def finalize(self):
        s = sorted(self.vals)
        if not s:
            return None
        mid = len(s) // 2
        return s[mid] if len(s) % 2 else (s[mid - 1] + s[mid]) / 2

con.create_aggregate("median", 1, Median)
print(con.execute("SELECT city, median(age) FROM people GROUP BY city ORDER BY city").fetchall())

class RunningSum:
    def __init__(self):
        self.total = 0
    def step(self, v):
        self.total += v
    def inverse(self, v):
        self.total -= v
    def value(self):
        return self.total
    def finalize(self):
        return self.total

if hasattr(con, 'create_window_function'):
    con.create_window_function("my_sum", 1, RunningSum)
    print(con.execute("""
        SELECT id, qty, my_sum(qty) OVER (ORDER BY id ROWS BETWEEN 1 PRECEDING AND CURRENT ROW)
        FROM orders ORDER BY id""").fetchall())
else:
    print('SKIP: custom window functions require Python 3.11+')

[(None, 25), ('Delhi', 32.5), ('Pune', 37.5)]
[(1, 10, 10), (2, 2, 12), (3, 5, 7), (4, 1, 6), (5, 20, 21), (6, 1, 21)]


## 11. create_collation: your own sort order
- A collation decides how TEXT compares and sorts. Built in: BINARY, NOCASE, RTRIM.
- `create_collation(name, func)` takes a function `(a, b)` returning -1, 0 or 1.
- Use it in `ORDER BY name COLLATE natsort` (not "natural": that is a SQL keyword), or on a column in `CREATE TABLE`.
- Example: natural sort so `file2` comes before `file10`.

In [13]:
def natural(a, b):
    key = lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]
    ka, kb = key(a), key(b)
    return (ka > kb) - (ka < kb)

con.create_collation("natsort", natural)
con.execute("CREATE TABLE files (name TEXT)")
con.executemany("INSERT INTO files VALUES (?)", [("file10",), ("file2",), ("File1",), ("file20",)])
print("binary :", [r[0] for r in con.execute("SELECT name FROM files ORDER BY name")])
print("natural:", [r[0] for r in con.execute("SELECT name FROM files ORDER BY name COLLATE natsort")])

binary : ['File1', 'file10', 'file2', 'file20']
natural: ['File1', 'file2', 'file10', 'file20']


## 12. set_trace_callback: see the SQL that really ran
- The trace callback receives every statement SQLite executes, with parameters filled in.
- Great for learning what a driver does behind the scenes (it shows the hidden `BEGIN` and `COMMIT`).
- Switch it off by passing `None`.

In [14]:
seen = []
t = sqlite3.connect(":memory:")                    # default isolation_level: driver manages BEGIN
t.set_trace_callback(seen.append)
t.execute("CREATE TABLE k (v INTEGER)")
t.execute("INSERT INTO k VALUES (?)", (7,))
t.execute("UPDATE k SET v = v + 1")
t.commit()
t.set_trace_callback(None)
for s in seen:
    print(s)
t.close()

CREATE TABLE k (v INTEGER)
BEGIN 
INSERT INTO k VALUES (7)
UPDATE k SET v = v + 1
COMMIT


## 13. set_authorizer: a gatekeeper for SQL
- The authorizer is called while a statement is compiled, once per action (read a column, delete, drop...).
- Return `sqlite3.SQLITE_OK` to allow, `SQLITE_DENY` to fail the whole statement, `SQLITE_IGNORE` to quietly treat it as NULL / skip.
- Useful when running SQL written by someone else, for example a read-only query sandbox.

In [15]:
def read_only(action, arg1, arg2, dbname, source):
    if action in (sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION):
        return sqlite3.SQLITE_OK
    return sqlite3.SQLITE_DENY

a = sqlite3.connect(":memory:")
a.execute("CREATE TABLE secrets (id INTEGER, note TEXT)")
a.execute("INSERT INTO secrets VALUES (1, 'hello')")
a.set_authorizer(read_only)
print(a.execute("SELECT note FROM secrets").fetchall())
for stmt in ("DELETE FROM secrets", "DROP TABLE secrets", "CREATE TABLE x (y)"):
    try:
        a.execute(stmt)
    except sqlite3.DatabaseError as e:
        print(type(e).__name__, "-", e)
# An allow-all callback also works on Python 3.10, which cannot clear with None.
a.set_authorizer(lambda *args: sqlite3.SQLITE_OK)
print(a.execute("SELECT COUNT(*) FROM secrets").fetchone())
a.close()

[('hello',)]
DatabaseError - not authorized
DatabaseError - not authorized
DatabaseError - not authorized
(1,)


## 14. set_progress_handler: stop runaway queries
- The handler is called every N virtual-machine steps while a query runs.
- Return a non-zero value to abort the query. SQLite raises `OperationalError: interrupted`.
- A simple way to put a work limit on user-supplied SQL. Pair it with the authorizer.

In [16]:
p = sqlite3.connect(":memory:")
ticks = {"n": 0}
def limit():
    ticks["n"] += 1
    return 1 if ticks["n"] > 50 else 0            # abort after 50 callbacks

p.set_progress_handler(limit, 1000)
endless = """WITH RECURSIVE c(x) AS (SELECT 1 UNION ALL SELECT x + 1 FROM c)
             SELECT COUNT(*) FROM c"""
try:
    p.execute(endless).fetchone()
except sqlite3.OperationalError as e:
    print(type(e).__name__, "-", e)
p.set_progress_handler(None, 0)
print(p.execute("SELECT 1 + 1").fetchone())
p.close()

OperationalError - interrupted
(2,)


## 15. Transactions from Python: with con, isolation_level, autocommit
- `with con:` opens a transaction, commits on success, rolls back if the block raises. It never closes the connection.
- `isolation_level` (default `""`) makes Python open a transaction before INSERT/UPDATE/DELETE. `None` means "do not touch transactions", so you write `BEGIN` yourself (that is what the notebook's `con` does).
- Python 3.12 adds `con.autocommit`: `True` (every statement commits), `False` (always in a transaction), or the legacy default (`-1`, shown below).
- `con.in_transaction` tells you whether a transaction is open right now.
- For what a transaction guarantees, see `06-Data-Modification` and `05-DDL-And-Integrity`.

In [17]:
t = sqlite3.connect(":memory:")
t.execute("CREATE TABLE acct (id INTEGER PRIMARY KEY, bal INTEGER CHECK (bal >= 0))")
t.execute("INSERT INTO acct VALUES (1, 100), (2, 50)")
t.commit()
print("in_transaction:", t.in_transaction, "| isolation_level:", repr(t.isolation_level), "| autocommit:", getattr(t, "autocommit", "legacy API (Python <3.12)"))

try:
    with t:
        t.execute("UPDATE acct SET bal = bal + 80 WHERE id = 2")
        t.execute("UPDATE acct SET bal = bal - 80 WHERE id = 1")   # 20, fine
        t.execute("UPDATE acct SET bal = bal - 80 WHERE id = 1")   # -60: CHECK fails
except sqlite3.IntegrityError as e:
    print("rolled back:", e)
print(t.execute("SELECT id, bal FROM acct ORDER BY id").fetchall())

t.execute("UPDATE acct SET bal = 0 WHERE id = 1")
print("in_transaction after UPDATE:", t.in_transaction)
t.rollback()
print("in_transaction after rollback:", t.in_transaction, t.execute("SELECT bal FROM acct WHERE id = 1").fetchone())
t.close()

in_transaction: False | isolation_level: '' | autocommit: -1
rolled back: CHECK constraint failed: bal >= 0
[(1, 100), (2, 50)]
in_transaction after UPDATE: True
in_transaction after rollback: False (100,)


In [18]:
# autocommit=True: every statement commits at once, you manage BEGIN yourself
if hasattr(sqlite3.Connection, "autocommit"):
    t = sqlite3.connect(":memory:", autocommit=True)
else:
    t = sqlite3.connect(":memory:", isolation_level=None)
    print("Python <3.12: isolation_level=None provides explicit transaction control")
t.execute("CREATE TABLE n (v INTEGER)")
t.execute("BEGIN")
t.execute("INSERT INTO n VALUES (1)")
print(t.in_transaction)
t.execute("ROLLBACK")
print(t.in_transaction, t.execute("SELECT COUNT(*) FROM n").fetchone())
t.close()

True
False (0,)


## 16. The error hierarchy
- `sqlite3.Error` is the base class. Under it: `DatabaseError`, and under that `IntegrityError`, `OperationalError`, `ProgrammingError`, `DataError`, `InternalError`, `NotSupportedError`.
- `IntegrityError`: a constraint failed (UNIQUE, NOT NULL, CHECK, FOREIGN KEY).
- `OperationalError`: the database could not do it (no such table, locked, bad SQL syntax).
- `ProgrammingError`: you used the API wrongly (wrong number of parameters, closed connection).
- Catch the narrowest class you can handle. Catch `sqlite3.Error` only at the edge of the program.

In [19]:
e1 = sqlite3.connect(":memory:")
e1.execute("PRAGMA foreign_keys = ON")
e1.execute("CREATE TABLE p (id INTEGER PRIMARY KEY, email TEXT UNIQUE NOT NULL)")
e1.execute("CREATE TABLE c (id INTEGER PRIMARY KEY, pid INTEGER REFERENCES p(id))")
e1.execute("INSERT INTO p VALUES (1, 'a@x.io')")
tests = [
    ("duplicate", "INSERT INTO p VALUES (2, 'a@x.io')", ()),
    ("null",      "INSERT INTO p VALUES (3, NULL)", ()),
    ("fk",        "INSERT INTO c VALUES (1, 99)", ()),
    ("no table",  "SELECT * FROM nope", ()),
    ("syntax",    "SELEC 1", ()),
    ("bindings",  "SELECT ?", ()),
]
for label, sql_text, params in tests:
    try:
        e1.execute(sql_text, params)
    except sqlite3.Error as e:
        print(f"{label:9} {type(e).__name__:17} {e}")
print([c.__name__ for c in sqlite3.IntegrityError.__mro__[:4]])
e1.close()

duplicate IntegrityError    UNIQUE constraint failed: p.email
null      IntegrityError    NOT NULL constraint failed: p.email
fk        IntegrityError    FOREIGN KEY constraint failed
no table  OperationalError  no such table: nope
syntax    OperationalError  near "SELEC": syntax error
bindings  ProgrammingError  Incorrect number of bindings supplied. The current statement uses 1, and there are 0 supplied.
['IntegrityError', 'DatabaseError', 'Error', 'Exception']


## 17. Threads and check_same_thread
- By default a connection may only be used by the thread that created it. Using it elsewhere raises `ProgrammingError`.
- Simplest safe design: one connection per thread (each opens its own).
- `check_same_thread=False` only switches the check off. You must still make sure two threads never use it at once, for example with a lock.
- An in-memory database is private to its connection, so threads sharing it need that one connection (or a file / shared-cache URI).

In [20]:
import threading
shared = sqlite3.connect(":memory:")
shared.execute("CREATE TABLE t (v INTEGER)")
out = {}
def worker():
    try:
        shared.execute("SELECT 1")
        out["ok"] = True
    except sqlite3.ProgrammingError as e:
        out["err"] = str(e)[:60]
th = threading.Thread(target=worker); th.start(); th.join()
print(out)
shared.close()

lock = threading.Lock()
loose = sqlite3.connect(":memory:", check_same_thread=False)
loose.execute("CREATE TABLE t (v INTEGER)")
def add(i):
    with lock, loose:
        loose.execute("INSERT INTO t VALUES (?)", (i,))
ths = [threading.Thread(target=add, args=(i,)) for i in range(5)]
for x in ths: x.start()
for x in ths: x.join()
print(loose.execute("SELECT COUNT(*), SUM(v) FROM t").fetchone())
loose.close()

{'err': 'SQLite objects created in a thread can only be used in that '}
(5, 10)


## 18. A file-backed database, safely
- Use `tempfile` for demos so nothing is left on disk. Always clean up.
- Two connections to the same file see committed data only. Uncommitted work stays private to its writer.
- `contextlib.closing` closes a connection when the block ends, which `with con:` does not do.

In [21]:
import tempfile, os, contextlib
d = tempfile.mkdtemp()
dbfile = os.path.join(d, "demo.db")
try:
    with contextlib.closing(sqlite3.connect(dbfile)) as w, contextlib.closing(sqlite3.connect(dbfile)) as r:
        w.execute("CREATE TABLE note (txt TEXT)")
        w.commit()
        w.execute("INSERT INTO note VALUES ('draft')")           # not committed yet
        print("reader before commit:", r.execute("SELECT COUNT(*) FROM note").fetchone())
        w.commit()
        print("reader after commit :", r.execute("SELECT COUNT(*) FROM note").fetchone())
finally:
    for f in os.listdir(d):
        os.remove(os.path.join(d, f))
    os.rmdir(d)
print("cleaned:", not os.path.exists(d))

reader before commit: (0,)
reader after commit : (1,)
cleaned: True


## 19. Blob I/O: read and write bytes in place
- A BLOB column holds raw bytes. For big values `con.blobopen(table, column, rowid)` gives a file-like handle (read, write, seek) without loading everything.
- A blob handle cannot change the size of the value. Make the row with `zeroblob(n)` first.
- Available from Python 3.11.

In [22]:
if hasattr(sqlite3.Connection, 'blobopen'):
    b = sqlite3.connect(":memory:")
    b.execute("CREATE TABLE files (id INTEGER PRIMARY KEY, data BLOB)")
    b.execute("INSERT INTO files VALUES (1, zeroblob(16))")
    with b.blobopen("files", "data", 1) as blob:
        blob.write(b"HELLO-")
        blob.seek(6)
        blob.write(b"SQLITE")
        print("length:", len(blob))
    print(b.execute("SELECT data, length(data), hex(substr(data, 1, 3)) FROM files").fetchone())
    with b.blobopen("files", "data", 1, readonly=True) as blob:
        print(blob.read(5))
    b.close()
else:
    print('SKIP: incremental BLOB I/O requires Python 3.11+')

length: 16
(b'HELLO-SQLITE\x00\x00\x00\x00', 16, '48454C')
b'HELLO'


## 20. Python and SQL: who does what
- Let SQL filter, join, group and sort. It is close to the data and can use indexes. Pulling everything into Python to loop is the common slow mistake.
- Let Python do what SQL is bad at: parsing, regex, calling web services, complex formatting, branching business rules.
- Middle road: push the heavy set work into SQL, then post-process the small result in Python (or register a small function).
- Bad vs good below: same answer, but the bad way moves every row into Python.

In [23]:
# bad: fetch everything, aggregate in Python
totals = {}
for pid, qty, price in con.execute("SELECT person_id, qty, price FROM orders"):
    totals[pid] = totals.get(pid, 0) + qty * price
print("python:", sorted(totals.items()))

# good: let SQL aggregate, Python only formats
rows = con.execute("""
    SELECT p.name, ROUND(SUM(o.qty * o.price), 2) AS total
    FROM orders o JOIN people p ON p.id = o.person_id
    GROUP BY p.id ORDER BY p.id""").fetchall()
print("sql   :", rows)

python: [(1, 39.0), (2, 7.5), (3, 58.0), (5, 12.5)]
sql   : [('Asha', 39.0), ('Bilal', 7.5), ('Chen', 58.0), ('Elif', 12.5)]


## 21. pandas: read_sql and to_sql
- `pandas.read_sql_query(sql, con, params=...)` turns a query result straight into a DataFrame.
- `df.to_sql(name, con, if_exists=...)` writes a DataFrame into a table.
- pandas may not be installed, so the import is guarded. Everything above works without it.
- A DataFrame is fine for analysis after the query. Do the filtering and grouping in SQL first so the frame stays small.

In [24]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("pandas not installed, skipping")
if pd is not None:
    df = pd.read_sql_query(
        "SELECT p.name, p.city, SUM(o.qty) AS items FROM people p "
        "LEFT JOIN orders o ON o.person_id = p.id GROUP BY p.id ORDER BY p.id", con)
    print(df.to_string(index=False))
    df.assign(items=df["items"].fillna(0).astype(int)).to_sql("item_counts", con, index=False, if_exists="replace")
    print(con.execute("SELECT * FROM item_counts ORDER BY name").fetchall())

 name  city  items
 Asha  Pune   12.0
Bilal Delhi    5.0
 Chen  Pune   21.0
 Dara   NaN    NaN
 Elif Delhi    1.0
[('Asha', 'Pune', 12), ('Bilal', 'Delhi', 5), ('Chen', 'Pune', 21), ('Dara', None, 0), ('Elif', 'Delhi', 1)]


## 22. Gotchas round-up
- The notebook `con` has `isolation_level=None`. A default connection opens transactions silently before writes, so remember to `commit()`.
- Rows are tuples by default: after you add a column to a `SELECT`, positional code breaks. Prefer `sqlite3.Row`.
- `cursor.rowcount` is -1 for SELECT and for some statements; trust it only for INSERT/UPDATE/DELETE.
- `lastrowid` is the id of the last inserted row on that cursor. It is unreliable after `executemany`.
- Registered adapters and converters are global to the process, but `create_function` and friends belong to one connection.

In [25]:
c = sqlite3.connect(":memory:")
c.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, v TEXT)")
cur = c.execute("INSERT INTO t (v) VALUES ('a')")
print("lastrowid:", cur.lastrowid, "rowcount:", cur.rowcount)
cur = c.execute("SELECT * FROM t")
print("select rowcount:", cur.rowcount)
cur = c.execute("UPDATE t SET v = 'z' WHERE id > 100")
print("update nothing rowcount:", cur.rowcount)
c.close()

lastrowid: 1 rowcount: 1
select rowcount: -1
update nothing rowcount: 0
